# Change log:
- Translator tool added to the handoff, which figures the language out from the email address's country code top-level domain (.hu)
- Agents are extended:
    - ComplAI Sales Manager sends out an email to IT4Lead company's CEO (new agent)
    - IT4Lead company's CEO asks his Head of Security (new agent) for his opinion
    - IT4Lead company's Head of Security sends his evaluation of the sales emal to CEO
    - IT4Lead company's CEO replies to the ComplAI Sales manager

- Technical details:
    - Tool is created about Sales Manager agent
    - IT4Lead company's CEO Head of Security  are tools as well
    - New agent `Business Life` agent was introduced which controls the workflow above
    - Mock email server was created (details below)

- How it can be improved:
    - instead of one cycle a more complex correspondence could be set

##### Mock Email Server

This version uses a local SQLite-backed mock mailbox instead of SendGrid.
- The agents can send, read, and reply to emails without creating real email accounts or using an external service.
- The mailbox data is persisted next to this notebook so you can inspect conversations between runs.
- There is no external setup needed for email in this version of the notebook.
- The helper file `mock_email_server.py` stores messages in a local SQLite database called `mock_email_server.sqlite3` in the same folder as this notebook.

That means your agents can:

- send emails to each other
- read inboxes by email address
- reply to existing emails with quoted history
- track which emails have been read

Everything stays local and is easy to inspect while you experiment with agent workflows.

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from typing import Dict
from mock_email_server import forward_email as forward_mock_email, mark_as_read as mark_mock_email_as_read, read_emails as read_mock_emails, reply_to_email as reply_mock_email, send_email as send_mock_email
import asyncio

In [2]:
load_dotenv(override=True)
LLM_MODEL = "gpt-4o-mini"

In [ ]:

COMPLAI_SALES_MANAGER_EMAIL = "sales.manager@complai.agent"
IT4LEAD_DOMAIN = "it4lead.hu" # Hungarian email address, so the Sales manager's email will be translated to Hungarian.
IT4LEAD_CEO_EMAIL = f"ceo@{IT4LEAD_DOMAIN}"
IT4LEAD_HO_SECURUTY_EMAIL = f"headof.security@{IT4LEAD_DOMAIN}"

In [4]:
@function_tool
def send_email(sender_email_address: str, recipient_email_address:str, subject: str, body: str) -> Dict[str, str]:
    """Send out an email with the given subject and HTML body to the mock recipient inbox."""
    return send_mock_email(
        sender_email_address,
        recipient_email_address,
        subject,
        body,
    )

In [5]:
@function_tool
def read_last_unread_email_from_recipient(owner_email_address:str, sender_email_address: str) -> dict | None:
    """Read the latest unread email sent to the mock recipient from a specific sender."""
    unread_emails = unread_emails = read_mock_emails(
        owner_email_address,
        unread_only=True,
        mark_as_read=False,
    )

    for email in reversed(unread_emails):
        if email["sender_email"] == sender_email_address:
            return mark_mock_email_as_read(email["id"])

    return None

In [6]:
@function_tool
def reply_to_email(email_id: int, sender_email_address: str, body: str) -> dict:
    """Reply to an existing mock email thread and append the original thread history."""
    return reply_mock_email(
        email_id,
        sender_email_address,
        body,
    )


In [7]:

@function_tool
def forward_email(email_id: int, sender_email_address: str, recipient_email_address: str) -> dict:
    """Forward an existing mock email while keeping the original email body."""
    return forward_mock_email(
        email_id,
        sender_email_address,
        recipient_email_address,
    )

# ComplAI Sales department

In [8]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [9]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=LLM_MODEL,
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=LLM_MODEL,
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=LLM_MODEL,
)

In [10]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)


In [11]:

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

translator_instructions = f"You are an expert to translate English text to other languages. \
You get the recipient email addess: {IT4LEAD_CEO_EMAIL}, the subject and email body. Based on the end of the email address detect the language you translate the text to. \
For example: x.y@gmbh.de -> German, y.z@vallalat.hu -> Hungarian, etc \
If you are not sure, do not translate the texts!"

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model=LLM_MODEL)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

translator = Agent(name="Multilanguage translator from English", instructions=translator_instructions, model=LLM_MODEL)
translator_tool = translator.as_tool(tool_name="translator", tool_description="Multilanguage translator from English")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model=LLM_MODEL)
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")


In [ ]:
handoff_tools = [subject_tool, html_tool, translator_tool, send_email]

In [13]:
instructions =instructions = f"""
You are an email translator, formatter, and sender.

You receive the plain-text body of an email to be sent to {IT4LEAD_CEO_EMAIL} from the {COMPLAI_SALES_MANAGER_EMAIL} email address.

Follow this exact sequence:
1. Use the `subject_writer` tool to create an email subject.
2. Because the recipient domain implies a non-English language, use the `translator` tool on both the subject and the body.
3. DO NOT Use the `html_converter` tool on the translated body.
4. Use `send_email` with the translated subject and translated HTML body.
Do not skip the `translator` step.
Do not send English text when translation is possible.
"""

emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=handoff_tools,
    model=LLM_MODEL,
    handoff_description="Convert an email to HTML and send it")


In [14]:
sales_manager_tools = [tool1, tool2, tool3]
sales_manager_handoffs = [emailer_agent]
print(sales_manager_tools)
print(sales_manager_handoffs)

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x77f2fc9451c0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x77f2e6704ae0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent3', description='Wr

In [ ]:
sales_manager_instructions = f"""
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.

Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.

2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.

3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
Make its sure that email is sent from {COMPLAI_SALES_MANAGER_EMAIL} email address!

Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=sales_manager_tools,
    handoffs=sales_manager_handoffs,
    model=LLM_MODEL)

sales_manager_as_tool = sales_manager.as_tool(tool_name="sales_manager", tool_description="Acting as Sales Manager of ComplAI")


# IT4Lead company

In [16]:
# Head of Security Agent
head_of_security_instructions = f"""You are the head of security at IT4Lead company, having decades of experience on this field.
Since your company provides B2B SaaS solutions to big enterprise companies as well among the several others, that is why the security is important, especially compliance.
Your CEO usually forward offers to you to evaluate it.
Follow these steps carefully:
1. Read you last email which was sent by the CEO from {IT4LEAD_CEO_EMAIL} email address to yours: {IT4LEAD_HO_SECURUTY_EMAIL}
2. Evaluate the offering of the security solution whether it would be interesting for your company. If it is not interesting give a reason. Head of Security has to clearly decide whether the company is interested or not.
3. Reply to is email sending your answer to the CEO from your email address {IT4LEAD_HO_SECURUTY_EMAIL} to his email {IT4LEAD_CEO_EMAIL}
"""

head_of_security_tools=[reply_to_email, read_last_unread_email_from_recipient]

head_of_security = Agent(
    name="Head of Security",
    instructions=head_of_security_instructions,
    tools=head_of_security_tools,
    model=LLM_MODEL
)

head_of_security_as_tool = head_of_security.as_tool(tool_name="security_opinion",tool_description="As the Head of Security review emails from CEO and answer them with sharing opinions about security")

In [17]:
# CEO Agent who receives the email and it will answer for that
ceo_instructions = f"""You are the CEO of the IT4Lead tech company which provides SaaS B2B solutions. You have big enterprise customers, that is why the security is important.
You get emails from potential vendors about their solutions. Since you have experts on this field you ask your experts and you expect answers from them. Based on their opinion you response to the vendors' sales manager email.
Follow these steps carefully:
1. Read the last unread email you got carefully. If you do not have unread email, just stop.
2. If your last email was received from {COMPLAI_SALES_MANAGER_EMAIL}, forward the email to your Head of Security's email address ({IT4LEAD_HO_SECURUTY_EMAIL}) asking his help for evaluating it. Make sure that the email contains the orignal content.
3. Use the Head of Security to read and answer to your email!
4. If your last email was received from {IT4LEAD_HO_SECURUTY_EMAIL}, read his recommendation about the offering and the send it to ComplAI's Sales Manager email address ({COMPLAI_SALES_MANAGER_EMAIL}) . Make sure that the email contains the orignal content.
"""
ceo_tools=[reply_to_email, forward_email, read_last_unread_email_from_recipient, head_of_security_as_tool]

ceo = Agent(
    name="CEO of IT4Lead",
    instructions=ceo_instructions,
    tools=ceo_tools,
    model=LLM_MODEL
)

ceo_as_mail_proxy_tool = ceo.as_tool(tool_name="ceo_as_mail_proxy", tool_description="Revceiving sales emails and ask the experts to evaluate them")

In [18]:
business_life_instructions = """You representing a simple sales flow in the business life. ComplAI company would like to sell their product, so the Sales Manager sends an email to IT4Lead company
Follow these steps carefully:
1. Sales Manager sends out a cold sales email addressed to Dear CEO from Alice
2. CEO of IT4Lead checks his emails and handle the message in it
"""

business_life_tools = [sales_manager_as_tool, ceo_as_mail_proxy_tool]

business_life =  Agent(
    name = "Business Life",
    instructions=business_life_instructions,
    tools=business_life_tools,
    model=LLM_MODEL
)

with trace("Sales Flow"):
    result = await Runner.run(business_life, "run the business sales flow")

# Test the results

In [23]:
# Read Sales Manager's email:
read_mock_emails(COMPLAI_SALES_MANAGER_EMAIL, mark_as_read=False)

[{'id': 36,
  'thread_id': 34,
  'sender_email': 'headof.security@it4lead.hu',
  'recipient_email': 'sales.manager@complai.agent',
  'subject': 'Re: Fwd: Tegye egyszerűbbé a SOC 2 megfelelőséget a ComplAI segítségével',
  'body': 'Kedves Alice,\n\nKöszönöm az ajánlatodat a ComplAI SOC 2 megfelelőség automatizálásával kapcsolatban! A megoldásod releváns a mi igényeink szempontjából, és úgy tűnik, hogy jelentős hatékonyságjavulást ígér az audit folyamatokban.\n\nJavaslom, hogy tartsunk egy megbeszélést, ahol részletesebben át tudjuk beszélni, hogyan tudnánk a ComplAI megoldását alkalmazni a szervezetünk specifikus igényeihez. Mikor lenne megfelelő számodra egy telefonhívás?\n\nÜdvözlettel,\n[Neved]\nVezérigazgató\nIT4Lead\n\n--- Original Message ---\nOn 2026-03-27T18:16:06.580418+00:00, sales.manager@complai.agent wrote:\n> Kedves [Név],\n> \n> Remélem, hogy ez az üzenet jól találja Önt. Az én nevem Alice, és a ComplAI értékesítési menedzsere vagyok, ahol a SOC 2 megfelelőség és audit el

In [24]:
# Read CEO's email:
read_mock_emails(IT4LEAD_CEO_EMAIL, mark_as_read=False)

[{'id': 22,
  'thread_id': 22,
  'sender_email': 'sales.manager@complai.agent',
  'recipient_email': 'ceo@it4lead.hu',
  'subject': 'Fedezze fel működését a ComplAI Insights segítségével',
  'body': '<!DOCTYPE html>\n<html lang="hu">\n<head>\n    <meta charset="UTF-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <style>\n        body {\n            font-family: Arial, sans-serif;\n            margin: 20px;\n            line-height: 1.5;\n            color: #333;\n        }\n        .container {\n            max-width: 600px;\n            margin: auto;\n            padding: 20px;\n            border: 1px solid #ddd;\n            border-radius: 5px;\n            background-color: #f9f9f9;\n        }\n        .header {\n            text-align: center;\n            margin-bottom: 20px;\n        }\n        .footer {\n            margin-top: 20px;\n            font-size: 0.9em;\n            color: #777;\n        }\n        .button {\n            display: 

In [25]:
# Read Head of Security's email:
read_mock_emails(IT4LEAD_HO_SECURUTY_EMAIL, mark_as_read=False)

[{'id': 23,
  'thread_id': 23,
  'sender_email': 'ceo@it4lead.hu',
  'recipient_email': 'headof.security@it4lead.hu',
  'subject': 'Fwd: Fedezze fel működését a ComplAI Insights segítségével',
  'body': '<!DOCTYPE html>\n<html lang="hu">\n<head>\n    <meta charset="UTF-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <style>\n        body {\n            font-family: Arial, sans-serif;\n            margin: 20px;\n            line-height: 1.5;\n            color: #333;\n        }\n        .container {\n            max-width: 600px;\n            margin: auto;\n            padding: 20px;\n            border: 1px solid #ddd;\n            border-radius: 5px;\n            background-color: #f9f9f9;\n        }\n        .header {\n            text-align: center;\n            margin-bottom: 20px;\n        }\n        .footer {\n            margin-top: 20px;\n            font-size: 0.9em;\n            color: #777;\n        }\n        .button {\n            displ

In [22]:
# Read the emails
# COMPLAI_SALES_MANAGER_EMAIL = "sales.manager@complai.agent"
# IT4LEAD_DOMAIN = "it4lead.hu"
# IT4LEAD_CEO_EMAIL = f"ceo@{IT4LEAD_DOMAIN}"
# IT4LEAD_HO_SECURUTY_EMAIL = f"headof.security@{IT4LEAD_DOMAIN}"
# read_mock_emails("test.recipient@t.it", mark_as_read=False)
read_mock_emails(IT4LEAD_CEO_EMAIL, mark_as_read=False)

[{'id': 22,
  'thread_id': 22,
  'sender_email': 'sales.manager@complai.agent',
  'recipient_email': 'ceo@it4lead.hu',
  'subject': 'Fedezze fel működését a ComplAI Insights segítségével',
  'body': '<!DOCTYPE html>\n<html lang="hu">\n<head>\n    <meta charset="UTF-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <style>\n        body {\n            font-family: Arial, sans-serif;\n            margin: 20px;\n            line-height: 1.5;\n            color: #333;\n        }\n        .container {\n            max-width: 600px;\n            margin: auto;\n            padding: 20px;\n            border: 1px solid #ddd;\n            border-radius: 5px;\n            background-color: #f9f9f9;\n        }\n        .header {\n            text-align: center;\n            margin-bottom: 20px;\n        }\n        .footer {\n            margin-top: 20px;\n            font-size: 0.9em;\n            color: #777;\n        }\n        .button {\n            display: 

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Can you identify the Agentic design patterns that were used here?<br/>
            What is the 1 line that changed this from being an Agentic "workflow" to "agent" under Anthropic's definition?<br/>
            Try adding in more tools and Agents! You could have tools that handle the mail merge to send to a list.<br/><br/>
            HARD CHALLENGE: add mock inbox tools so agents can read and reply to each other's emails,
            Then have the SDR keep the conversation going inside the local mailbox. This may require some "vibe coding" 😂
            </span>
        </td>
    </tr>
</table>